# 🧪 Lab 2: Three Old-School JSON Techniques (The Security Gates Before VARIANT)

In this laboratory session, we audit the legacy security gates that Apache Spark historically used to process semi-structured JSON payloads before the advent of the native VARIANT data type. We will implement and analyze three traditional techniques: raw string extraction, strict parsing using an explicit schema via `from_json`, and schema inference using `schema_of_json`. Through this process, we will document the exact engineering trade-offs, limitations, and failure modes of each historical approach.

### Step 1: Initialize Local Spark Workspace
We spin up our local standalone Spark session and establish our test data structures.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

# Locate or spin up our local standalone Spark instance
spark = SparkSession.builder.getOrCreate()

print(f"Active Spark Engine Version: {spark.version}")

Active Spark Engine Version: 4.1.2


### Step 2: Track 1 — Raw String Ingestion & Path Excavation
We store our incoming payload entirely as an unparsed JSON string and use `get_json_object` to perform a quick-and-dirty field excavation.

In [2]:
# Simulated bronze landing layer: raw JSON text trapped inside a string column
raw_string_data = [
    ("BKG-001", '{"origin":"MAD","destination":"BCN","carrier":"IB","fare_family":"EconomyLight","totalAmount":87.50,"currency":"EUR","baggage":{"included":true},"passenger_count":1}'),
    ("BKG-002", '{"origin":"CDG","destination":"JFK","carrier":"AF","fare_family":"EconomyClassic","totalAmount":520.00,"currency":"USD","baggage":{"included":true},"passenger_count":2}')
]

string_df = spark.createDataFrame(raw_string_data, ["booking_id", "raw_payload"])

# Excavate several nested fields from the same payload row
excavated_df = string_df.select(
    "booking_id",
    F.get_json_object(F.col("raw_payload"), "$.origin").alias("origin"),
    F.get_json_object(F.col("raw_payload"), "$.destination").alias("destination"),
    F.get_json_object(F.col("raw_payload"), "$.carrier").alias("carrier"),
    F.get_json_object(F.col("raw_payload"), "$.fare_family").alias("fare_family"),
    F.get_json_object(F.col("raw_payload"), "$.totalAmount").alias("total_amount"),
    F.get_json_object(F.col("raw_payload"), "$.currency").alias("currency"),
    F.get_json_object(F.col("raw_payload"), "$.baggage.included").alias("baggage_included"),
    F.get_json_object(F.col("raw_payload"), "$.passenger_count").alias("passenger_count")
)

print("=== TRACK 1: RAW STRING EXCAVATION LIST ===")
excavated_df.show(truncate=False)
excavated_df.printSchema()

=== TRACK 1: RAW STRING EXCAVATION LIST ===
+----------+------+-----------+-------+--------------+------------+--------+----------------+---------------+
|booking_id|origin|destination|carrier|fare_family   |total_amount|currency|baggage_included|passenger_count|
+----------+------+-----------+-------+--------------+------------+--------+----------------+---------------+
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.5        |EUR     |true            |1              |
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.0       |USD     |true            |2              |
+----------+------+-----------+-------+--------------+------------+--------+----------------+---------------+

root
 |-- booking_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- fare_family: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- currency: string (nullable = true)
 

### Step 3: Track 2 — Strict Parsing with an Explicit Contract
We implement the strict gate using `from_json` alongside an explicitly declared schema contract to parse our payloads once into structural complex columns.

In [3]:
# Define an explicit schema contract matching the expected flight booking layout
explicit_booking_schema = T.StructType([
    T.StructField("origin", T.StringType(), True),
    T.StructField("destination", T.StringType(), True),
    T.StructField("carrier", T.StringType(), True),
    T.StructField("fare_family", T.StringType(), True),
    T.StructField("totalAmount", T.DecimalType(10, 2), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("baggage", T.StructType([T.StructField("included", T.BooleanType(), True)]), True),
    T.StructField("passenger_count", T.IntegerType(), True)
])

parsed_strict_df = string_df.withColumn("parsed_struct", F.from_json(F.col("raw_payload"), explicit_booking_schema))

clean_serving_df = parsed_strict_df.select(
    "booking_id",
    F.col("parsed_struct.origin").alias("origin"),
    F.col("parsed_struct.destination").alias("destination"),
    F.col("parsed_struct.carrier").alias("carrier"),
    F.col("parsed_struct.fare_family").alias("fare_family"),
    F.col("parsed_struct.totalAmount").alias("total_amount"),
    F.col("parsed_struct.currency").alias("currency"),
    F.col("parsed_struct.baggage.included").alias("baggage_included"),
    F.col("parsed_struct.passenger_count").alias("passenger_count")
)

print("=== TRACK 2: CLEAN serving LAYER FROM STRICT PARSER ===")
clean_serving_df.show(truncate=False)

=== TRACK 2: CLEAN serving LAYER FROM STRICT PARSER ===
+----------+------+-----------+-------+--------------+------------+--------+----------------+---------------+
|booking_id|origin|destination|carrier|fare_family   |total_amount|currency|baggage_included|passenger_count|
+----------+------+-----------+-------+--------------+------------+--------+----------------+---------------+
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.50       |EUR     |true            |1              |
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.00      |USD     |true            |2              |
+----------+------+-----------+-------+--------------+------------+--------+----------------+---------------+



### Step 4: Track 2b — Injecting Controlled Ingestion Damage
We build an anomalous, drifting dataset containing syntax damage, missing fields, alternative types, and uninvited partner metadata to analyze the exact failure boundaries of the explicit gate.

In [4]:
damaged_payloads = [
    ("BKG-MALFORMED", '{"origin":"MAD", "destination": "BCN",'), # Malformed syntax case
    ("BKG-MISSING-BAG", '{"origin":"MAD","destination":"BCN","carrier":"IB","fare_family":"EconomyLight","totalAmount":87.50,"currency":"EUR","passenger_count":1}'), # Absent baggage field
    ("BKG-TYPE-DRIFT", '{"origin":"MAD","destination":"BCN","carrier":"IB","fare_family":"EconomyLight","totalAmount":"149.99","currency":"EUR","baggage":{"included":true},"passenger_count":1}'), # Numeric field as text
    ("BKG-UNINVITED-EXT", '{"origin":"MAD","destination":"BCN","carrier":"IB","fare_family":"EconomyLight","totalAmount":87.50,"currency":"EUR","baggage":{"included":true},"passenger_count":1,"supplier_metadata":{"uninvited_system_debug_flag":"true"}}') # Unexpected extension cases
]

damaged_string_df = spark.createDataFrame(damaged_payloads, ["booking_id", "raw_payload"])

damaged_parsed_df = damaged_string_df.withColumn("parsed_struct", F.from_json(F.col("raw_payload"), explicit_booking_schema))

print("=== TRACK 2B: ANOMALOUS FOOTPRINT RADAR ===")
damaged_parsed_df.select(
    "booking_id",
    F.col("parsed_struct.origin").alias("origin"),
    F.col("parsed_struct.totalAmount").alias("total_amount"),
    F.col("parsed_struct.baggage.included").alias("baggage_included"),
    F.col("parsed_struct").alias("full_extracted_struct")
).show(truncate=False)

=== TRACK 2B: ANOMALOUS FOOTPRINT RADAR ===
+-----------------+------+------------+----------------+----------------------------------------------------+
|booking_id       |origin|total_amount|baggage_included|full_extracted_struct                               |
+-----------------+------+------------+----------------+----------------------------------------------------+
|BKG-MALFORMED    |NULL  |NULL        |NULL            |{NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL}    |
|BKG-MISSING-BAG  |MAD   |87.50       |NULL            |{MAD, BCN, IB, EconomyLight, 87.50, EUR, NULL, 1}   |
|BKG-TYPE-DRIFT   |MAD   |149.99      |true            |{MAD, BCN, IB, EconomyLight, 149.99, EUR, {true}, 1}|
|BKG-UNINVITED-EXT|MAD   |87.50       |true            |{MAD, BCN, IB, EconomyLight, 87.50, EUR, {true}, 1} |
+-----------------+------+------------+----------------+----------------------------------------------------+



### Step 5: Track 3 — Schema Inference Volatility Analysis
We use `schema_of_json` on a basic economy-only data sample, and then witness how it falls short when hitting a rich supplier sample that contains optional business branches.

In [5]:
# Base economy-only profile sample
base_sample = '{"origin":"MAD","destination":"BCN","totalAmount":87.50}'

# Rich business-class profile sample containing an unexpected penalty structure extension
rich_sample = '{"origin":"MAD","destination":"BCN","totalAmount":520.00,"refund_penalty":{"amount":150.00,"currency":"EUR"},"corporate_contract":"CORP-777"}'

inferred_base_schema = spark.range(1).select(F.schema_of_json(F.lit(base_sample))).collect()[0][0]
inferred_rich_schema = spark.range(1).select(F.schema_of_json(F.lit(rich_sample))).collect()[0][0]

print("=== TRACK 3: SCHEMA INFERENCE COMPARISON REPORT ===")
print(f"Inferred Schema from Economy Base Sample:\n{inferred_base_schema}\n")
print(f"Inferred Schema from Rich Business Sample:\n{inferred_rich_schema}")

=== TRACK 3: SCHEMA INFERENCE COMPARISON REPORT ===
Inferred Schema from Economy Base Sample:
STRUCT<destination: STRING, origin: STRING, totalAmount: DOUBLE>

Inferred Schema from Rich Business Sample:
STRUCT<corporate_contract: STRING, destination: STRING, origin: STRING, refund_penalty: STRUCT<amount: DOUBLE, currency: STRING>, totalAmount: DOUBLE>


### Step 6: Audit Execution Blueprints & Physical Plans
We output the execution plans for both older approaches to trace the repeated string scanning overhead.

In [6]:
print("=== PHYSICAL PLAN: TRACK 1 (REPETITIVE STRINGS) ===")
excavated_df.explain()

print("\n=== PHYSICAL PLAN: TRACK 2 (SINGLE PARSE STRUCT) ===")
clean_serving_df.explain()

=== PHYSICAL PLAN: TRACK 1 (REPETITIVE STRINGS) ===
== Physical Plan ==
*(1) Project [booking_id#0, get_json_object(raw_payload#1, $.origin) AS origin#2, get_json_object(raw_payload#1, $.destination) AS destination#3, get_json_object(raw_payload#1, $.carrier) AS carrier#4, get_json_object(raw_payload#1, $.fare_family) AS fare_family#5, get_json_object(raw_payload#1, $.totalAmount) AS total_amount#6, get_json_object(raw_payload#1, $.currency) AS currency#7, get_json_object(raw_payload#1, $.baggage.included) AS baggage_included#8, get_json_object(raw_payload#1, $.passenger_count) AS passenger_count#9]
+- *(1) Scan ExistingRDD[booking_id#0,raw_payload#1]



=== PHYSICAL PLAN: TRACK 2 (SINGLE PARSE STRUCT) ===
== Physical Plan ==
*(2) Project [booking_id#0, parsed_struct#38.origin AS origin#39, parsed_struct#38.destination AS destination#40, parsed_struct#38.carrier AS carrier#41, parsed_struct#38.fare_family AS fare_family#42, parsed_struct#38.totalAmount AS total_amount#43, parsed_struct

## 📊 Post-Lab Analysis: Evidence from the Engine Room

Evaluating our legacy security gates directly isolates the computational and structural trade-offs that forced the development of Spark 4's native `VARIANT` architecture.

### 1. The Raw String Parsing Subscription Plan
* **Repetitive Scan Overheads:** Reviewing our physical blueprints under Step 6 exposes the structural penalty tax of raw text queries. When running Track 1, the query execution plan registers a separate `get_json_object` operator mapping for every single field call.
* **Opaque Text Suitcases:** Because Spark treats the row entirely as an unparsed text suitcase, it must unpack and scan the exact same string payload nine distinct times per row to fetch our variables.
* **The String-Shaped Curse:** String excavation fails to provide Spark with structural type awareness. The engine forces every single excavated column to default to a loose, non-reconciled `string (nullable = true)` type regardless of whether the original literal was an unquoted number or a primitive boolean.

### 2. Permissive Boundaries & Dropped Extras (The Security Radar)
* **The Type-Drift Coercion:** Our Track 2b experiment with anomalous payloads under Step 4 proves that Spark's default permissive parsing mode has flexible coercion limits. When `totalAmount` arrived as a quoted string text wrapper (`"149.99"`), Spark's `from_json` parser successfully performed an automatic type conversion to a clean numeric decimal.
* **Silent Architecture Amnesia:** Conversely, the uninvited `supplier_metadata` payload block was dropped completely from memory. Because our explicit schema contract did not actively declare it, Spark quietly deleted the new feature branch without generating an error flag.
* **The Governance Trade-off:** This silent dropping of fields creates a dangerous architecture gap where your upstream partner could upgrade their schema versions mid-flight without your silver serving tables ever recording the change.

### 3. The `DOUBLE` Precision Trap (The Flashlight Flaw)
* **The Sample Blindspot:** Our Track 3 structural bootstrap comparison under Step 5 documents why sample-based schema discovery fails as a governance model. When the inference flashlight evaluated only our economy baseline, the resulting schema completely missed our rich business-class extensions and penalty loops.
* **Floating-Point Improvisation:** Because automated inference only knows what it witnesses, it incorrectly typed our financial `totalAmount` metrics as a floating-point **`DOUBLE`**. In commercial aviation data, monetary values require exact fixed-point `DECIMAL` scale; floating-point doubles introduce rounding volatility that ruins audit ledgers.
* **The Structural Floor:** Relying on dynamic inference means your table schemas shift structural shapes row-by-row based on provider drift. This turning of your production data contract into a volatile weather forecast forces downstream dashboards into a permanent state of silent sadness.
